# Video Diffusion Model — Kaggle Training

Trains the custom 3D U-Net video diffusion model from scratch on Kaggle's free T4/P100 GPU (16 GB).  
No pretrained weights, no external dataset — everything is self-contained.

**Before running:**
1. Enable GPU: *Settings → Accelerator → GPU T4 x2* (or P100)
2. Enable Internet: *Settings → Internet → On*
3. **If repo is private:** add a `GITHUB_TOKEN` secret in *Kaggle → Settings → Secrets*
4. Set `HF_TOKEN` secret if you want HuggingFace checkpoint sync (optional)

In [ ]:
# ── 0. Config ─────────────────────────────────────────────────────────────────
GITHUB_USER   = "aniketpandav"
GITHUB_REPO   = "video-model"          # repo name only, no .git
REPO_DIR      = "/kaggle/working/Video-Model"
CONFIG        = "configs/train_kaggle.yaml"
HF_REPO_ID    = ""        # e.g. "aniketpandav/vdm-kaggle" — leave blank to skip
SMOKE_TEST    = True      # always True first — 200 steps ~3 min, confirms everything works
RESUME_CKPT   = ""        # path to last.pt to resume; leave blank for fresh start

In [ ]:
# ── 1. System info ────────────────────────────────────────────────────────────
import subprocess, sys, os

def run(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, check=check, text=True,
                            capture_output=True, cwd=cwd)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    return result

run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
run("pip install -q imageio imageio-ffmpeg opencv-python-headless tqdm pyyaml huggingface_hub")

In [ ]:
# ── 3. Clone repo (handles both public and private repos) ─────────────────────
from kaggle_secrets import UserSecretsClient

# Try to get GitHub token (needed only if repo is private)
try:
    gh_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    clone_url = f"https://{gh_token}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    print("Using GitHub token (private repo mode)")
except Exception:
    clone_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    print("No GITHUB_TOKEN secret found — trying public clone")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already cloned — pulling latest")
    run(f"git -C {REPO_DIR} pull")
else:
    run(f"git clone {clone_url} {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working dir : {os.getcwd()}")
run("ls")

In [ ]:
# ── 4. Smoke test (200 steps, ~3 min) — run this first every time ─────────────
if SMOKE_TEST:
    print("Running smoke test (200 steps)...")
    result = run(f"python scripts/train.py --config {CONFIG} --steps 200", cwd=REPO_DIR, check=False)
    if result.returncode == 0:
        print("\nSmoke test PASSED. Set SMOKE_TEST=False and re-run cell 0 + cell 5 for full training.")
    else:
        print(f"\nSmoke test FAILED (exit {result.returncode}). Fix the error above before full training.")
else:
    print("Smoke test skipped.")

In [ ]:
# ── 5. Full training run (~20-25 h on T4 for 80k steps) ──────────────────────
# Only run this after the smoke test passes.
# Checkpoints save every 2000 steps to /kaggle/working/runs/kaggle/
# They persist as Kaggle output even if the session expires.

if SMOKE_TEST:
    print("SMOKE_TEST is still True — set it to False in cell 0 before running full training.")
else:
    resume_flag = f"--resume {RESUME_CKPT}" if RESUME_CKPT else ""
    train_cmd = f"python scripts/train.py --config {CONFIG} {resume_flag}"
    print(f"Starting: {train_cmd}")
    proc = subprocess.run(train_cmd, shell=True, cwd=REPO_DIR)
    print("Training finished" if proc.returncode == 0 else f"Exited with code {proc.returncode}")

In [ ]:
# ── 6. Generate sample GIF from checkpoint ───────────────────────────────────
ckpt_path = "/kaggle/working/runs/kaggle/last.pt"
out_path  = "/kaggle/working/sample_final.gif"

if os.path.exists(ckpt_path):
    run(f"python scripts/sample.py --ckpt {ckpt_path} --n 4 --steps 100 --out {out_path}",
        cwd=REPO_DIR)
    from IPython.display import Image, display
    display(Image(filename=out_path))
else:
    print(f"No checkpoint at {ckpt_path} yet")

In [ ]:
# ── 7. (Optional) Sync checkpoint to HuggingFace Hub ─────────────────────────
# Set HF_REPO_ID in cell 0 and add HF_TOKEN to Kaggle Secrets.

if HF_REPO_ID:
    from huggingface_hub import HfApi
    try:
        token = UserSecretsClient().get_secret("HF_TOKEN")
        api = HfApi(token=token)
        api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
        api.upload_folder(
            folder_path="/kaggle/working/runs/kaggle",
            repo_id=HF_REPO_ID, repo_type="model",
            commit_message="Kaggle checkpoint"
        )
        print(f"Synced to https://huggingface.co/{HF_REPO_ID}")
    except Exception as e:
        print(f"HF sync failed: {e}")
else:
    print("HF_REPO_ID not set — checkpoint stays in /kaggle/working/runs/kaggle/")

In [ ]:
# ── 8. List output files ──────────────────────────────────────────────────────
out_dir = "/kaggle/working/runs/kaggle"
if os.path.isdir(out_dir):
    for f in sorted(os.listdir(out_dir)):
        size = os.path.getsize(os.path.join(out_dir, f)) / 1e6
        print(f"  {f:45s} {size:6.1f} MB")
else:
    print("Output dir not found yet.")